# Food Biotechnology Laboratory Course  
**Course Code:** 752-5004-00L  

## Save the Jupyter Notebook

1. Look at the top-left corner of Colab. If you see “☁️ Cannot save changes”, it means your notebook isn’t saved to your Google Drive yet.

2. Click on File → Save a copy in Drive.

3. Your notebook will now be saved in your Google Drive, and the message should change to “Saved to Drive”.

💡 Tip for students: Always check the top-left corner to make sure your work is being saved automatically.

## Sourdough Production and Fermentation Analysis  
### Metabolomics Data Analysis with Python

This laboratory tutorial introduces **primary untargeted metabolomics analysis** using a **two-factor experimental design**. The case study focuses on **sourdough fermentation** and examines how **microbial activity shapes metabolite profiles** across fermentation time and different cereal substrates.

---

## Learning Objectives

By the end of this laboratory session, you will be able to:

- Load and inspect metabolomics **feature tables** and **sample metadata**
- Clean, transform, scale, and normalize **mass spectrometry peak area data**
- Compare metabolite profiles across cereal substrates  
  (*wheat, spelt, buckwheat, maize*)
- Compare metabolite profiles across fermentation time points  
  (*t00 / t01* vs *t7*)

---

## Statistical Analysis in Untargeted Metabolomics

The primary goal of statistical analysis in metabolomics is the **categorization, comparison, and prediction of sample properties**. The validity of statistical models depends strongly on upstream analytical and computational choices.

### Data Quality and Pre-processing

The reliability of statistical inference depends critically on:

- Data acquisition parameters  
- Feature detection and filtering  
- Missing-value imputation  
- Data transformation, scaling, and normalization  

These steps determine variance structure, feature weighting, and downstream biological interpretation.

### Core Variables in Mass Spectrometry–Based Metabolomics

In mass spectrometry–based metabolomics, each detected feature is defined by two key variables:

- **m/z (mass-to-charge ratio)**  
- **Signal intensity** (peak area or abundance)

Together, these variables form the quantitative basis for all downstream statistical analyses.

---

## Statistical Analysis Strategies

Untargeted metabolomics data are typically explored using both **univariate** and **multivariate** statistical approaches. These methods provide complementary perspectives, and reliance on a single model or visualization is insufficient for robust interpretation.

### Univariate Analysis

Univariate analysis evaluates **one metabolite at a time**, usually through statistical testing between experimental conditions.

A common visualization of univariate results is:

- **Volcano plots** - display fold change versus statistical significance (e.g. p-value), with each metabolite evaluated independently.


### Multivariate Analysis

Multivariate analysis operates on the **full metabolite matrix**, analyzing all metabolites simultaneously. These methods capture covariance structures and reveal global patterns across samples.

Typical applications include:
- Dimensionality reduction  
- Sample clustering  
- Classification and prediction  

Common multivariate methods:

- **Principal Component Analysis (PCA)** — unsupervised dimensionality reduction  
- **Partial Least Squares Discriminant Analysis (PLS-DA)** — dimensionality reduction    
- **Heatmaps** — multivariate visualization of metabolite abundance patterns across samples

---
## Knowledge-Based Analysis

Metabolomics data can also be interpreted using **knowledge-based approaches** that integrate statistical results with biochemical information.

- **Pathway enrichment analysis**


Knowledge-based interpretation is affected by analytical bias introduced by LC–MS workflows. Thus, pathway-level conclusions must be interpreted in the context of analytical coverage and experimental design.

## 1. Setup


In this section, we prepare the Python environment required for the metabolomics data analysis. This includes installing the necessary software packages and importing the libraries that will be used throughout the laboratory course.


### Library Imports

After installation, the required libraries are imported into the Python environment. Each library plays a specific role in the metabolomics workflow:

- **pandas** and **numpy**  
  Used for loading, cleaning, and processing metabolomics feature tables and metadata.

- **matplotlib**, **seaborn**, and **plotly**  
  Used for generating visualizations such as PCA plots, heatmaps, and volcano plots.

- **scikit-learn**  
  Provides tools for data preprocessing (e.g., scaling and imputation) and multivariate analysis, including **Principal Component Analysis (PCA)**.

- **scipy** and **statsmodels**  
  Used for statistical testing, including univariate analyses such as *t-tests*.

- **ipywidgets** and **IPython display utilities**  
  Enable interactive elements and improved display of figures within Jupyter notebooks.

- **warnings**  
  Used to suppress non-critical warnings to keep the output clean and readable.

In [ ]:
# =========================
# Package Installation (Colab)
# =========================
# Colab may already include most of these
!pip install pandas numpy matplotlib seaborn scikit-learn statsmodels plotly ipywidgets sspa pubchempy datasets

# =========================
# Core Libraries
# =========================
import warnings
import re
import requests
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import patches, gridspec, cm
from matplotlib.patches import Patch
import seaborn as sns
import plotly.express as px

# =========================
# SciPy & Statistics
# =========================
from scipy import stats
from scipy.stats import ttest_ind
from scipy.cluster.hierarchy import linkage, dendrogram

# =========================
# Scikit-learn (ML & Preprocessing)
# =========================
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# =========================
# Pathway Enrichment
# =========================
import sspa

# =========================
# Interactive Widgets & Display
# =========================
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# =========================
# Google Colab Utilities
# =========================
from google.colab import files

# =========================
# Hugging Face Datasets
# =========================
from datasets import load_dataset, utils

## 2. Load Data

In this step, we upload the metabolomics data and load it into Python for analysis. Two CSV files are required: a **feature table** containing metabolite intensities and a **metadata table** describing the experimental samples. In Google Colab, a file selection dialog will appear to upload these files.

### Metadata Table

The metadata table contains sample-level information such as **flour type** and **fermentation time point**. Each row corresponds to one sample, and the sample names must match the column names in the feature table.


### Feature Table

The feature table was created using mzMine contains the quantitative metabolomics data. Rows represent **metabolite features** (defined by feature number, m/z and retention time), and columns represent **samples**. Feature intensities correspond to mass spectrometry peak areas.

### Purpose

This step ensures that the metabolomics measurements and experimental metadata are correctly loaded and aligned, providing the foundation for downstream preprocessing, statistical analysis, and visualization.


In [ ]:
# Upload CSVs (Colab will prompt file selection)
print("-------------------------------------------------------")
print("Upload feature table (rows=metabolites, columns=samples)")
print("-------------------------------------------------------")
feature_file = files.upload()

print("-------------------------------------------------------")
print("Upload metadata table")
print("-------------------------------------------------------")
metadata_file = files.upload()

# Convert uploaded files to DataFrames
feature_df = pd.read_csv(list(feature_file.keys())[0], index_col=0)
metadata_df = pd.read_csv(list(metadata_file.keys())[0], index_col=0)

# Extract 'InchiKey' as a separate vector and remove it from the feature table
if 'InchiKey' in feature_df.columns:
    inchkeys = feature_df['InchiKey'].copy()
    feature_df = feature_df.drop(columns=['InchiKey'])
else:
    inchi_vector = None

-------------------------------------------------------
Upload feature table (rows=metabolites, columns=samples)
-------------------------------------------------------


Saving 260227_featureTable_SourDough.csv to 260227_featureTable_SourDough (1).csv
-------------------------------------------------------
Upload metadata table
-------------------------------------------------------


Saving 260227_metaData_SourDough.csv to 260227_metaData_SourDough (1).csv


## 3. Inspect Data

In [ ]:
print("Feature table shape:", feature_df.shape)
feature_df.head()

Feature table shape: (5151, 112)


,260226_Hpos_000_AnalyticalBlnk.mzML,260226_Hpos_001_AnalyticalBlnk.mzML,260226_Hpos_002_AnalyticalBlnk.mzML,260226_Hpos_005_MethodBlnk.mzML,260226_Hpos_006_MethodBlnk.mzML,260226_Hpos_007_QC_cond.mzML,260226_Hpos_008_QC_cond.mzML,260226_Hpos_009_QC_cond.mzML,260226_Hpos_010_QC_cond.mzML,260226_Hpos_017_QC.mzML,...,260226_Hpos_110_maize_t8_F.mzML,260226_Hpos_111_maize_t1_F.mzML,260226_Hpos_112_wheat_t1_H.mzML,260226_Hpos_113_wheat_t1_G.mzML,260226_Hpos_114_maize_t8_G.mzML,260226_Hpos_115_wheat_t8_B.mzML,260226_Hpos_116_maize_t8_H.mzML,260226_Hpos_117_buckwheat_t1_F.mzML,260226_Hpos_118_wheat_t1_A.mzML,260226_Hpos_119_QC.mzML
1 | 494.8120mz | 0.14min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,60.61,1116.0,NaN,NaN,NaN,NaN,NaN
2 | 150.0915mz | 0.08min,45.84,9040.0,6546.0,15910.0,3866.0,13860.0,8708.0,2586.0,13910.0,7073.0,...,2137.0,5465.0,12960.0,10180.00,5169.0,4946.0,3758.0,10590.0,7753.0,3242.0
3 | 1-(2-Phenylethyl)urea | [M + H]+,2100.00,37800.0,18550.0,40000.0,3784.0,26180.0,16350.0,39940.0,16320.0,37960.0,...,30210.0,5438.0,31530.0,21720.00,33020.0,23360.0,29410.0,33800.0,29940.0,31360.0
6 | 537.5356mz | 0.34min,2078.00,12790.0,22960.0,1601.0,3107.0,1428.0,2461.0,1153.0,1586.0,6020.0,...,4969.0,5702.0,3002.0,6256.00,3476.0,1341.0,1431.0,5745.0,603.0,3280.0
"7 | 4-(3,4-Xylyl)-3-thiosemicarbazide | [M + H]+",169300.00,154200.0,78150.0,68800.0,71070.0,69930.0,69160.0,67260.0,68070.0,69620.0,...,62450.0,14820.0,30260.0,21350.00,58900.0,71450.0,64790.0,63750.0,16270.0,63840.0


In [ ]:
print("Metadata table shape:", metadata_df.shape)
metadata_df.head()
metadata_df


Metadata table shape: (112, 7)


,run_order,sample_type,condition,timepoint,group,condition_timepoint,condition_timepoint_group
file_name,,,,,,,
260226_Hpos_000_AnalyticalBlnk.mzML,0,Analytical Blank,Analytical Blank,NaN,NaN,NaN,NaN
260226_Hpos_001_AnalyticalBlnk.mzML,1,Analytical Blank,Analytical Blank,NaN,NaN,NaN,NaN
260226_Hpos_002_AnalyticalBlnk.mzML,2,Analytical Blank,Analytical Blank,NaN,NaN,NaN,NaN
260226_Hpos_005_MethodBlnk.mzML,5,Method Blank,Method Blank,NaN,NaN,NaN,NaN
260226_Hpos_006_MethodBlnk.mzML,6,Method Blank,Method Blank,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
260226_Hpos_115_wheat_t8_B.mzML,115,Sample,wheat,8.0,B,wheat_8,wheat_8_B
260226_Hpos_116_maize_t8_H.mzML,116,Sample,maize,8.0,H,maize_8,maize_8_H
260226_Hpos_117_buckwheat_t1_F.mzML,117,Sample,buckwheat,1.0,F,buckwheat_1,buckwheat_1_F


## 4. Volcano Plot for Differential Metabolite Analysis

Volcano plots are a widely used **univariate analysis** method in metabolomics. They allow simultaneous visualization of the **magnitude of change** (fold change) and **statistical significance** for individual metabolites when comparing two experimental groups.

---

### Analysis Workflow

1. **Group Selection**
   - Select **two distinct experimental groups** (`group1` vs `group2`) for comparison (e.g. different cereals or fermentation time points).

2. **Univariate Statistical Computation**
   Each metabolite is analyzed independently:
   - **Log2 fold change (log2FC)**: difference in mean metabolite abundance between the two groups.
   - **P-value**: calculated using a two-sample (independent) t-test to assess statistical significance.
   - Missing values (`NaN`) are excluded on a metabolite-by-metabolite basis.

3. **Visualization**
   - Scatter plot with:
     - **log2FC on the x-axis**
     - **−log10(p-value) on the y-axis**
   - Points are colored according to statistical outcome:
     - Red: significantly higher in group 2
     - Blue: significantly lower in group 2
     - Grey: not statistically significant
   - Horizontal and vertical lines indicate p-value and fold-change thresholds.
   - Individual metabolites can be highlighted for closer inspection.

---

### Important Considerations

- Volcano plots are **exploratory** and should not be treated as a primary analysis.
- Use **raw or minimally processed data** for volcano plot calculations.
- Do **not** use log-transformed, centered, or scaled data prepared for multivariate analysis, as this can distort fold changes and p-values.

In [ ]:
class VolcanoPlotter:
    def __init__(self, feature_table, metadata_df, group_col="condition_timepoint"):
        self.master_feature_table = feature_table.copy()
        self.feature_table = feature_table.copy()
        self.metadata = metadata_df
        self.group_col = group_col
        self.metadata.index = self.metadata.index.astype(str)
        self.feature_table.columns = self.feature_table.columns.astype(str)

        self.current_df = None
        self.xlims = None
        self.ylims = None
        self.header = 'log2(FC) | -log10(p-value)'
        self.last_groups = (None, None)

    def show_loading(self, output_widget):
        """Displays a CSS spinner in the output area."""
        spinner_html = """
        <div style="display: flex; align-items: center; justify-content: center; height: 200px; flex-direction: column;">
            <div class="loader"></div>
            <p style="font-family: sans-serif; margin-top: 15px;">Calculating statistics... Please wait.</p>
            <style>
                .loader {
                  border: 8px solid #f3f3f3;
                  border-top: 8px solid #3498db;
                  border-radius: 50%;
                  width: 50px;
                  height: 50px;
                  animation: spin 1s linear infinite;
                }
                @keyframes spin {
                  0% { transform: rotate(0deg); }
                  100% { transform: rotate(360deg); }
                }
            </style>
        </div>
        """
        with output_widget:
            clear_output(wait=True)
            display(HTML(spinner_html))

    def apply_filter(self, filter_type):
        if filter_type == "Only Annotated":
            mask = (
                self.master_feature_table.index.astype(str).str.contains('mz', case=False, na=False) &
                self.master_feature_table.index.astype(str).str.contains('min', case=False, na=False)
            )
            self.feature_table = self.master_feature_table[~mask].copy()
        else:
            self.feature_table = self.master_feature_table.copy()
        self.current_df = None

    def calculate_stats(self, attr_col, group1, group2):
        samples_g1 = self.metadata[self.metadata[attr_col].astype(str) == str(group1)].index
        samples_g2 = self.metadata[self.metadata[attr_col].astype(str) == str(group2)].index
        cols_g1 = self.feature_table.columns.intersection(samples_g1)
        cols_g2 = self.feature_table.columns.intersection(samples_g2)

        if len(cols_g1) < 2 or len(cols_g2) < 2:
            return None

        results = []
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for metabolite in self.feature_table.index:
                s1 = self.feature_table.loc[metabolite, cols_g1].dropna()
                s2 = self.feature_table.loc[metabolite, cols_g2].dropna()
                if len(s1) < 2 or len(s2) < 2: continue

                _, p_val = stats.ttest_ind(s1, s2, nan_policy='omit')
                fc = np.log2((np.mean(s2) + 1e-9) / (np.mean(s1) + 1e-9))

                if not np.isnan(p_val) and not np.isnan(fc):
                    results.append({'Metabolite': metabolite, 'p_val': p_val, 'log2FC': fc})

        df = pd.DataFrame(results)
        if not df.empty:
            df['-log10p'] = -np.log10(df['p_val'].replace(0, 1e-9))
            x_max = df['log2FC'].abs().max() * 1.05
            y_max = df['-log10p'].max() * 1.05
            self.xlims = (-x_max, x_max)
            self.ylims = (0, y_max)
        return df if not df.empty else None

    def plot(self, attr_col, group1, group2, highlight_id=None, p_thresh=0.05, fc_thresh=1.0):
        if group1 == group2:
            print("Select two different groups.")
            return

        df = self.current_df
        if df is None:
            print("No data available for these groups/filter.")
            return

        fig, ax = plt.subplots(figsize=(10, 6))
        up = df[(df['p_val'] < p_thresh) & (df['log2FC'] > fc_thresh)]
        down = df[(df['p_val'] < p_thresh) & (df['log2FC'] < -fc_thresh)]
        ns = df[(df['p_val'] >= p_thresh) | (df['log2FC'].abs() <= fc_thresh)]

        ax.scatter(ns['log2FC'], ns['-log10p'], c='lightgrey', alpha=0.4, label='Not Significant', s=20)
        ax.scatter(up['log2FC'], up['-log10p'], c='red', alpha=0.6, label=f'Up in {group2}', s=25)
        ax.scatter(down['log2FC'], down['-log10p'], c='blue', alpha=0.6, label=f'Down in {group2}', s=25)

        if highlight_id and highlight_id != self.header:
            match = df[df['Metabolite'] == highlight_id]
            if not match.empty:
                m_data = match.iloc[0]
                ax.scatter(m_data['log2FC'], m_data['-log10p'], c='yellow', edgecolors='black', s=120, zorder=10)
                ax.annotate(highlight_id, (m_data['log2FC'], m_data['-log10p']),
                            xytext=(12, 12), textcoords='offset points', fontweight='bold',
                            bbox=dict(boxstyle='round,pad=0.3', fc='yellow', alpha=0.7))

        ax.set_xlim(self.xlims)
        ax.set_ylim(self.ylims)
        ax.axvline(fc_thresh, color='black', linestyle='--', alpha=0.3)
        ax.axvline(-fc_thresh, color='black', linestyle='--', alpha=0.3)
        ax.axhline(-np.log10(p_thresh), color='black', linestyle='--', alpha=0.3)
        ax.set_title(f"Volcano Plot: {group1} vs {group2}", fontsize=14)
        ax.set_xlabel("Log2 Fold Change")
        ax.set_ylabel("-Log10 P-value")
        ax.legend(loc='upper right')
        plt.tight_layout()
        plt.show()

    def interactive(self):
        attr_options = sorted(self.metadata.columns.tolist())
        w_filter = widgets.Dropdown(options=["All Features", "Only Annotated"], description="Filter:")
        w_attr = widgets.Dropdown(options=attr_options, value=self.group_col, description="Attribute")

        def get_group_options(attr):
            return sorted([str(x) for x in self.metadata[attr].unique() if pd.notnull(x)])

        initial_vals = get_group_options(w_attr.value)
        w_g1 = widgets.Dropdown(options=initial_vals, value=initial_vals[0], description="Group 1")
        w_g2 = widgets.Dropdown(options=initial_vals, value=initial_vals[1] if len(initial_vals) > 1 else initial_vals[0], description="Group 2")
        w_metab = widgets.Dropdown(options=[(self.header, self.header)], value=self.header, description="Inspect:")

        output = widgets.Output()

        def update_metab_options():
            if self.current_df is not None:
                sorted_df = self.current_df.sort_values('log2FC', ascending=False)
                options = [(self.header, self.header)] + [
                    (f"log2FC={r['log2FC']:.2f} | -log10P={r['-log10p']:.2f} | {r['Metabolite']}", r['Metabolite'])
                    for _, r in sorted_df.iterrows()
                ]
                w_metab.options = options
            else:
                w_metab.options = [(self.header, self.header)]
            w_metab.value = self.header

        def run_plot_only(change=None):
            """Just refreshes the plot area."""
            with output:
                clear_output(wait=True)
                self.plot(w_attr.value, w_g1.value, w_g2.value, highlight_id=w_metab.value)

        def on_group_or_filter_change(change):
            """Triggered when parameters change that require re-calculation."""
            # 1. Show Spinner
            self.show_loading(output)

            # 2. Perform background work
            self.apply_filter(w_filter.value)
            self.current_df = self.calculate_stats(w_attr.value, w_g1.value, w_g2.value)
            self.last_groups = (w_g1.value, w_g2.value)

            # 3. Update dropdown and show final plot
            update_metab_options()
            run_plot_only()

        def on_attr_change(change):
            new_vals = get_group_options(change['new'])
            w_g1.options, w_g2.options = new_vals, new_vals
            on_group_or_filter_change(None)

        w_filter.observe(on_group_or_filter_change, names='value')
        w_attr.observe(on_attr_change, names='value')
        w_g1.observe(on_group_or_filter_change, names='value')
        w_g2.observe(on_group_or_filter_change, names='value')
        w_metab.observe(run_plot_only, names='value')

        display(widgets.VBox([
            widgets.HBox([w_filter, w_attr]),
            widgets.HBox([w_g1, w_g2]),
            widgets.HBox([w_metab]),
            output
        ]))
        on_group_or_filter_change(None)

# Run the visualizer
volcano = VolcanoPlotter(feature_df, metadata_df)
volcano.interactive()

## 5. Data Preparation for Statistical Analysis

The processing of feature table, ensuring the dataset is suitable for statistical and multivariate analyses such as **PCA**, **PLS-DA**, **heatmaps**, and other statistical or machine learning approaches.  

### Key Preprocessing Steps

1. **Zero / Missing Value Handling**  
   - `zero_strategy="replace"` → Replace zeros or NA values with a small constant (default: 1e-10).  
   - `zero_strategy="nan"` → Convert zeros to NA while retaining missingness.  
   - `zero_strategy=None` → Leave data unchanged.

2. **Coefficient of Variation (CV) Filtering**  
   - Remove features with high variability due to technical noise.  
   - `cv_threshold` sets the cutoff (e.g., 0.3).  
   - CV can be calculated on **raw intensities** or **log-transformed values**.

3. **Imputation of Missing Values**  
   - `row_mean` → Replace missing values with the feature-wise mean (default).  
   - `row_median` → Robust to outliers.  
   - `knn` → Uses k-nearest neighbors to impute missing entries.  
   - `None` → No imputation performed.  

4. **Log Transformation**  
   - `log_transform=True` → Apply log(x + 1) transformation to stabilize variance and reduce skewness.  
   - Essential in metabolomics to normalize distributions and simplify fold-change interpretation.

5. **Scaling**  
   - `zscore` → Mean-centered, unit variance (default).  
   - `pareto` → Mean-centered, variance dampened (square root of standard deviation).  
   - `None` → No scaling applied.  
   - Scaling ensures all features contribute equally to variance-sensitive analyses such as PCA and PLS-DA.

### Important Notes

- There is **no one-size-fits-all preprocessing pipeline**; the choice of steps depends on the experimental design and downstream analyses.  
- Avoid over-processing: excessive transformations may distort biological signals.  
- Preprocessing aligns the dataset with the assumptions of multivariate methods:
  - PCA, PLS-DA, and heatmaps require **complete, scaled, and optionally log-transformed data**.  
  - CAVE: Volcano plots or univariate analyses  use **raw or minimally processed data**.  


In [ ]:
def process_feature_table(feature_df,
                          zero_strategy="replace",
                          zero_value=1e-10,
                          cv_threshold=0.0,
                          cv_on="raw",
                          impute_method="row_mean",
                          log_transform=True,
                          scale="zscore"):
    """
    Preprocess feature table while preserving row and column IDs.
    Rows = samples (filenames), Columns = features (m/z_RT)
    """
    df = feature_df.copy()

    # --------------------
    # Zero / NA Handling
    # --------------------
    if zero_strategy == "replace":
        df = df.replace(0, np.nan).fillna(zero_value)
    elif zero_strategy == "nan":
        df = df.replace(0, np.nan)

    # --------------------
    # CV Filtering
    # --------------------
    if cv_threshold > 0:
        cv_base = np.log1p(df) if cv_on == "log" else df
        cv = cv_base.std(axis=0) / cv_base.mean(axis=0).replace(0, np.nan)
        keep_cols = cv.index[cv >= cv_threshold]
        df = df[keep_cols]

    # --------------------
    # Imputation
    # --------------------
    if impute_method == "row_mean":
        df = df.fillna(df.mean(axis=0))
    elif impute_method == "row_median":
        df = df.fillna(df.median(axis=0))
    elif impute_method == "knn":
        from sklearn.impute import KNNImputer
        imputer = KNNImputer()
        df = pd.DataFrame(imputer.fit_transform(df), index=df.index, columns=df.columns)

    # --------------------
    # Log Transform
    # --------------------
    if log_transform:
        df = pd.DataFrame(np.log1p(df), index=df.index, columns=df.columns)

    # --------------------
    # Scaling
    # --------------------
    if scale in ["zscore", "pareto"]:
        df_mean = df.mean(axis=0)
        df_std = df.std(axis=0).replace(0, np.nan)
        if scale == "zscore":
            df_scaled = (df - df_mean) / df_std
        else:  # pareto
            df_scaled = (df - df_mean) / np.sqrt(df_std)
        df = pd.DataFrame(df_scaled, index=df.index, columns=df.columns)

    return df


In [ ]:
processed_df = process_feature_table(feature_df,
                                     zero_strategy="replace",
                                     cv_threshold=0.3,
                                     log_transform=True,
                                     scale="zscore")

print("Processed feature table shape:", processed_df.shape)
processed_df.head()


Processed feature table shape: (5151, 112)


,260226_Hpos_000_AnalyticalBlnk.mzML,260226_Hpos_001_AnalyticalBlnk.mzML,260226_Hpos_002_AnalyticalBlnk.mzML,260226_Hpos_005_MethodBlnk.mzML,260226_Hpos_006_MethodBlnk.mzML,260226_Hpos_007_QC_cond.mzML,260226_Hpos_008_QC_cond.mzML,260226_Hpos_009_QC_cond.mzML,260226_Hpos_010_QC_cond.mzML,260226_Hpos_017_QC.mzML,...,260226_Hpos_110_maize_t8_F.mzML,260226_Hpos_111_maize_t1_F.mzML,260226_Hpos_112_wheat_t1_H.mzML,260226_Hpos_113_wheat_t1_G.mzML,260226_Hpos_114_maize_t8_G.mzML,260226_Hpos_115_wheat_t8_B.mzML,260226_Hpos_116_maize_t8_H.mzML,260226_Hpos_117_buckwheat_t1_F.mzML,260226_Hpos_118_wheat_t1_A.mzML,260226_Hpos_119_QC.mzML
1 | 494.8120mz | 0.14min,-0.309775,-0.330010,-0.330980,-0.474883,-0.480489,-1.244433,-1.237666,-1.241060,-1.264355,-1.248519,...,-1.022663,-1.162119,-1.091962,0.022794,0.644159,-1.126292,-0.999128,-1.278956,-1.087782,-1.256244
2 | 150.0915mz | 0.08min,1.380779,3.464142,3.311363,2.274943,1.859604,1.009192,0.903261,0.614840,1.004834,0.851773,...,0.762556,0.802949,1.049955,1.231808,0.990442,0.825944,0.905755,0.829766,0.924228,0.663661
3 | 1-(2-Phenylethyl)urea | [M + H]+,3.052301,4.059978,3.743099,2.536970,1.853533,1.159473,1.051919,1.261221,1.042836,1.249866,...,1.379157,0.801819,1.251040,1.411180,1.409507,1.182192,1.381800,1.093815,1.227743,1.202535
6 | 537.5356mz | 0.34min,3.047675,3.608657,3.831507,1.622428,1.797704,0.472277,0.605115,0.424187,0.488502,0.813585,...,0.958956,0.812641,0.719195,1.116569,0.900787,0.526536,0.682433,0.690629,0.350819,0.666428
"7 | 4-(3,4-Xylyl)-3-thiosemicarbazide | [M + H]+",4.981283,4.645549,4.339237,2.691110,2.684336,1.391639,1.392251,1.384308,1.382509,1.393573,...,1.548229,1.030707,1.241742,1.407113,1.540296,1.438758,1.564561,1.238183,1.090738,1.371348


## 6. Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is an unsupervised multivariate technique that reduces the dimensionality of metabolomics data while retaining the primary sources of variation. It converts correlated metabolite variables into a smaller set of uncorrelated principal components (PCs), helping to uncover patterns among samples, time points, or experimental conditions.

### PCA Workflow

- Select a pair of PCs for the X-axis and Y-axis (e.g., PC1 vs PC2)  
- Choose a metadata attribute to color or group samples  
- Filter for specific groups or values to focus the visualization  

### Notes on PCA

- Complete numeric data is required; preprocessing (e.g., normalization, scaling) is essential  

### Interpretation

- Samples that cluster together have similar metabolite profiles  
- Separation along PCs may reflect differences in experimental factors such as flour type or fermentation time  
- Interactive PCA exploration allows analysis of:  
  - Sample grouping by metadata  
  - Variance explained by each PC  
  - Global patterns and relationships in metabolomics datasets

In [ ]:
class PCAVisualizer:
    NATURE_30_COLORS = [
        "#1F77B4", "#D62728", "#2CA02C", "#9467BD", "#FF7F0E",
        "#17BECF", "#8C564B", "#E377C2", "#7F7F7F", "#BCBD22",
        "#AEC7E8", "#FF9896", "#98DF8A", "#C5B0D5", "#FFBB78",
        "#9EDAE5", "#C49C94", "#F7B6D2", "#C7C7C7", "#DBDB8D",
        "#393B79", "#637939", "#8C6D31", "#843C39", "#7B4173",
        "#3182BD", "#E6550D", "#31A354", "#756BB1", "#636363",
        "#6BAED6", "#FD8D3C", "#74C476", "#9E9AC8", "#969696"
    ]

    def __init__(self, data, metadata, sample_col="file_name", group_col="condition_timepoint", n_components=6):
        self.data = data.T
        self.metadata = metadata.copy()
        self.sample_col = sample_col
        self.group_col = group_col
        self.n_components = n_components

        if self.sample_col not in self.metadata.columns:
            self.metadata = self.metadata.reset_index()

        self._run_pca()

    def _run_pca(self):
        self.pca = PCA(n_components=self.n_components)
        self.pcs = self.pca.fit_transform(self.data.values)
        self.pc_df = pd.DataFrame(
            self.pcs,
            index=self.data.index,
            columns=[f"PC{i+1}" for i in range(self.n_components)]
        )

    def show_loading(self, output_widget):
        spinner_html = """
        <div style="display: flex; align-items: center; justify-content: center; height: 200px; flex-direction: column;">
            <div class="loader"></div>
            <p style="font-family: sans-serif; margin-top: 15px;">Rendering PCA... Please wait.</p>
            <style>
                .loader {
                  border: 8px solid #f3f3f3;
                  border-top: 8px solid #2ecc71;
                  border-radius: 50%;
                  width: 50px;
                  height: 50px;
                  animation: spin 1s linear infinite;
                }
                @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
            </style>
        </div>
        """
        with output_widget:
            clear_output(wait=True)
            display(HTML(spinner_html))

    @staticmethod
    def _confidence_ellipse(x, y, ax, confidence=0.90, **kwargs):
        if len(x) < 3: return
        cov = np.cov(x, y)
        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:, order]
        chi2_val = 4.605
        width, height = 2 * np.sqrt(vals * chi2_val)
        angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        ellipse = patches.Ellipse(xy=(np.mean(x), np.mean(y)), width=width, height=height,
                                  angle=angle, fill=False, **kwargs)
        ax.add_patch(ellipse)

    def plot(self, pc_x, pc_y, attr_col, selected_values):
        meta_indexed = self.metadata.set_index(self.sample_col)
        common_samples = self.pc_df.index.intersection(meta_indexed.index)

        temp_df = self.pc_df.loc[common_samples].copy()
        temp_df[attr_col] = meta_indexed.loc[common_samples, attr_col].astype(str).values

        fig, ax = plt.subplots(figsize=(10, 7))
        plot_df = temp_df[temp_df[attr_col].isin(selected_values)]

        # Keep selected_values order (which is now numerically sorted)
        unique_vals = [v for v in selected_values if v in plot_df[attr_col].unique()]

        num_groups = len(unique_vals)
        if num_groups > len(self.NATURE_30_COLORS):
            viridis_cmap = plt.get_cmap("viridis", num_groups)
            color_map = {val: viridis_cmap(i) for i, val in enumerate(unique_vals)}
        else:
            color_map = {val: self.NATURE_30_COLORS[i] for i, val in enumerate(unique_vals)}

        for val in unique_vals:
            subset = plot_df[plot_df[attr_col] == val]
            if subset.empty: continue

            ax.scatter(subset[pc_x], subset[pc_y], label=str(val),
                       color=color_map[val], s=120, edgecolors='white', linewidth=1, alpha=0.8)

            self._confidence_ellipse(subset[pc_x], subset[pc_y], ax,
                                     edgecolor=color_map[val], linewidth=2, linestyle='--')

        ix_x, ix_y = int(pc_x[2:])-1, int(pc_y[2:])-1
        ax.set_xlabel(f"{pc_x} ({self.pca.explained_variance_ratio_[ix_x]*100:.1f}%)", fontweight='bold')
        ax.set_ylabel(f"{pc_y} ({self.pca.explained_variance_ratio_[ix_y]*100:.1f}%)", fontweight='bold')
        ax.set_title(f"PCA grouped by {attr_col}", fontsize=14, pad=15)

        ax.legend(title=attr_col, bbox_to_anchor=(1.05, 1), loc='upper left',
                  frameon=True, fontsize='small' if num_groups > 20 else 'medium')

        ax.grid(True, linestyle=':', alpha=0.6)
        plt.tight_layout()
        plt.show()

    def interactive(self):
        pc_options = [f"PC{i+1}" for i in range(self.n_components)]
        attr_options = [c for c in self.metadata.columns if c != self.sample_col]

        w_pc_x = widgets.Dropdown(options=pc_options, value="PC1", description="X-axis")
        w_pc_y = widgets.Dropdown(options=pc_options, value="PC2", description="Y-axis")
        w_attr = widgets.Dropdown(options=attr_options, value=self.group_col, description="Attribute")

        def get_safe_options(attr):
            """Intelligently sorts options numerically if they are digits."""
            vals = self.metadata[attr].dropna().unique()
            str_vals = [str(v) for v in vals]

            try:
                # Attempt to sort numerically
                return sorted(str_vals, key=lambda x: float(x))
            except ValueError:
                # Fallback to alphanumeric sort if non-numeric strings exist
                return sorted(str_vals)

        initial_opts = get_safe_options(w_attr.value)
        w_vals = widgets.SelectMultiple(
            options=initial_opts,
            value=tuple(initial_opts),
            description='Values',
            layout=widgets.Layout(width='400px', height='200px')
        )

        output = widgets.Output()

        def run_update(change=None):
            self.show_loading(output)
            with output:
                clear_output(wait=True)
                self.plot(w_pc_x.value, w_pc_y.value, w_attr.value, w_vals.value)

        def on_attr_change(change):
            new_opts = get_safe_options(change['new'])
            w_vals.options = new_opts
            w_vals.value = tuple(new_opts)
            run_update()

        w_pc_x.observe(run_update, names='value')
        w_pc_y.observe(run_update, names='value')
        w_vals.observe(run_update, names='value')
        w_attr.observe(on_attr_change, names='value')

        ui = widgets.VBox([
            widgets.HBox([w_pc_x, w_pc_y]),
            widgets.HBox([w_attr, w_vals]),
            output
        ])

        display(ui)
        run_update()

# Run the visualizer
pca_vis = PCAVisualizer(data=processed_df, metadata=metadata_df)
pca_vis.interactive()

## 7. Partial Least Squares Discriminant Analysis (PLS-DA)

Partial Least Squares Discriminant Analysis (PLS-DA) is a supervised multivariate method that models differences between predefined sample groups. Unlike PCA, PLS-DA uses class labels to maximize group separation.

### Method Overview

- **X matrix**: metabolite intensities (samples × features)  
- **Y vector**: class membership (binary: 1 = selected class, 0 = all others)  
- A PLS regression model is fitted, and latent variables (LVs) are extracted  

### Visualization and Interpretation

- Scores plots (e.g., LV1 vs LV2) illustrate class separation  
- Confidence ellipses summarize within-class variability  
- Clear separation indicates systematic metabolite differences; overlap suggests weak discrimination  

### Important Notes

- PLS-DA is supervised and can be prone to overfitting  
- Visual separation alone does not confirm statistical significance  


In [ ]:
class PLSDAVisualizer:
    def __init__(self, feature_table, metadata, sample_col="file_name", group_col="condition_timepoint", n_components=2):
        # 1. Median Normalization
        self.master_feature_table = feature_table.copy()
        sample_medians = self.master_feature_table.median()
        self.master_feature_table = self.master_feature_table.div(sample_medians, axis=1)

        self.feature_table = self.master_feature_table.copy()
        self.metadata = metadata.copy()
        self.sample_col = sample_col
        self.group_col = group_col
        self.n_components = n_components

        if self.sample_col not in self.metadata.columns:
            self.metadata = self.metadata.reset_index()

    def show_loading(self, output_widget):
        spinner_html = """
        <div style="display: flex; align-items: center; justify-content: center; height: 200px; flex-direction: column;">
            <div class="loader"></div>
            <p style="font-family: sans-serif; margin-top: 15px;">Validating & Calculating VIPs... Please wait.</p>
            <style>
                .loader {
                  border: 8px solid #f3f3f3;
                  border-top: 8px solid #3498db;
                  border-radius: 50%;
                  width: 50px;
                  height: 50px;
                  animation: spin 1s linear infinite;
                }
                @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
            </style>
        </div>
        """
        with output_widget:
            clear_output(wait=True)
            display(HTML(spinner_html))

    def _calculate_vips(self, model):
        """Calculates VIP scores for the features."""
        t, w, q = model.x_scores_, model.x_weights_, model.y_loadings_
        p, h = w.shape
        vips = np.zeros((p,))
        s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
        total_s = np.sum(s)
        for i in range(p):
            weight = np.array([(w[i, j] / np.linalg.norm(w[:, j]))**2 for j in range(h)])
            vips[i] = np.sqrt(p * (s.T @ weight) / total_s)
        return vips

    def plot(self, lv_x, lv_y, attr_col, group1, group2, top_n_vips=15):
        if group1 == group2:
            print("Select two different groups.")
            return

        meta_sub = self.metadata[self.metadata[attr_col].isin([group1, group2])]
        samples = meta_sub[self.sample_col].values
        common_cols = self.feature_table.columns.intersection(samples)

        if len(common_cols) < 10:
            print("Insufficient samples for validation. 10+ samples recommended.")
            return

        X = self.feature_table[common_cols].T.values
        y_labels = meta_sub.set_index(self.sample_col).loc[common_cols, attr_col]
        y = (y_labels == group2).astype(int).values.reshape(-1, 1)

        # Train/Test Split to prevent overfitting
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        pls = PLSRegression(n_components=self.n_components)
        pls.fit(X_train_scaled, y_train)

        # Metrics for Overfitting Check
        y_train_pred = pls.predict(X_train_scaled)
        y_test_pred = pls.predict(X_test_scaled)
        r2_train = r2_score(y_train, y_train_pred)
        r2_test = r2_score(y_test, y_test_pred)

        # Plotting Data
        X_all_scaled = scaler.transform(X)
        scores = pls.transform(X_all_scaled)
        vips = self._calculate_vips(pls)

        score_df = pd.DataFrame(scores, index=common_cols, columns=[f"LV{i+1}" for i in range(self.n_components)])
        score_df["Class"] = y_labels.values

        # Three-panel Visualization
        fig = plt.figure(figsize=(22, 7))
        gs = fig.add_gridspec(1, 3)
        ax1 = fig.add_subplot(gs[0, 0]) # Scores
        ax2 = fig.add_subplot(gs[0, 1]) # Validation
        ax3 = fig.add_subplot(gs[0, 2]) # VIPs

        # 1. Score Plot
        colors = {group1: "#e74c3c", group2: "#3498db"}
        for cls, subset in score_df.groupby("Class"):
            ax1.scatter(subset[lv_x], subset[lv_y], s=130, label=cls,
                        color=colors.get(cls, "gray"), edgecolors="white", alpha=0.8)
            self._confidence_ellipse(subset[lv_x], subset[lv_y], ax1, edgecolor=colors.get(cls, "gray"))

        ax1.set_title(f"Score Plot\nTrain R² = {r2_train:.3f} | Test Q² = {r2_test:.3f}", fontweight='bold')
        ax1.set_xlabel(lv_x, fontweight='bold')  # X-axis label
        ax1.set_ylabel(lv_y, fontweight='bold')  # Y-axis label
        ax1.legend()
        ax1.grid(True, linestyle=":", alpha=0.6)

        # 2. Prediction vs Observation Plot
        ax2.scatter(y_train, y_train_pred, color='gray', alpha=0.4, label='Train')
        ax2.scatter(y_test, y_test_pred, color='orange', s=100, edgecolors='black', label='Test', zorder=5)
        ax2.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Threshold')
        ax2.set_xlabel("Observed Label", fontweight='bold')
        ax2.set_ylabel("Predicted Value", fontweight='bold')
        ax2.set_title("Validation: Predicted vs Observed", fontweight='bold')
        ax2.legend()

        # 3. VIP Plot
        vip_df = pd.DataFrame({'Feature': self.feature_table.index, 'VIP': vips})
        vip_df = vip_df.sort_values(by='VIP', ascending=True).tail(top_n_vips)
        ax3.barh(vip_df['Feature'], vip_df['VIP'], color='#16a085', alpha=0.8)
        ax3.axvline(1, color='red', linestyle='--', label='VIP > 1')
        ax3.set_title(f'Top {top_n_vips} VIP Features', fontweight='bold')
        ax3.set_xlabel("VIP Score", fontweight='bold')  # X-axis label
        ax3.set_ylabel("Feature", fontweight='bold')    # Y-axis label
        ax3.yaxis.tick_right()
        ax3.yaxis.set_label_position("right")

        plt.tight_layout()
        plt.show()

    def apply_filter(self, filter_type):
        if filter_type == "Only Annotated":
            mask = (self.master_feature_table.index.astype(str).str.contains('mz', case=False) &
                    self.master_feature_table.index.astype(str).str.contains('min', case=False))
            self.feature_table = self.master_feature_table[~mask].copy()
        else:
            self.feature_table = self.master_feature_table.copy()

    @staticmethod
    def _confidence_ellipse(x, y, ax, confidence=0.90, **kwargs):
        if len(x) < 3: return
        cov = np.cov(x, y)
        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:, order]
        chi2_val = 4.605
        width, height = 2 * np.sqrt(vals * chi2_val)
        angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        ellipse = patches.Ellipse(xy=(np.mean(x), np.mean(y)), width=width, height=height,
                                  angle=angle, fill=False, linestyle="--", linewidth=2, **kwargs)
        ax.add_patch(ellipse)

    def interactive(self):
        lv_options = [f"LV{i+1}" for i in range(self.n_components)]
        attr_options = sorted([c for c in self.metadata.columns if c != self.sample_col])
        w_filter = widgets.Dropdown(options=["All Features", "Only Annotated"], description="Filter:")
        w_attr = widgets.Dropdown(options=attr_options, value=self.group_col, description="Attribute:")

        def get_group_options(attr):
            return sorted([str(x) for x in self.metadata[attr].unique() if pd.notnull(x)])

        initial_groups = get_group_options(w_attr.value)
        w_g1 = widgets.Dropdown(options=initial_groups, value=initial_groups[0], description="Group 1:")
        w_g2 = widgets.Dropdown(options=initial_groups, value=initial_groups[1] if len(initial_groups) > 1 else initial_groups[0], description="Group 2:")
        w_lv_x = widgets.Dropdown(options=lv_options, value="LV1", description="X-axis:")
        w_lv_y = widgets.Dropdown(options=lv_options, value="LV2", description="Y-axis:")
        w_top_n = widgets.IntSlider(value=15, min=5, max=100, step=1, description="Top VIPs:")

        output = widgets.Output()
        def run_update(change=None):
            self.show_loading(output)
            self.apply_filter(w_filter.value)
            with output:
                clear_output(wait=True)
                self.plot(w_lv_x.value, w_lv_y.value, w_attr.value, w_g1.value, w_g2.value, w_top_n.value)

        def on_attr_change(change):
            new_groups = get_group_options(change['new'])
            w_g1.options, w_g2.options = new_groups, new_groups
            if len(new_groups) > 1: w_g1.value, w_g2.value = new_groups[0], new_groups[1]
            run_update()

        for w in [w_filter, w_g1, w_g2, w_lv_x, w_lv_y, w_top_n]: w.observe(run_update, names='value')
        w_attr.observe(on_attr_change, names='value')

        display(widgets.VBox([
            widgets.HBox([w_filter, w_attr]),
            widgets.HBox([w_g1, w_g2]),
            widgets.HBox([w_lv_x, w_lv_y, w_top_n]),
            output
        ]))
        run_update()

# Run the class
plsda_vis = PLSDAVisualizer(feature_table=processed_df, metadata=metadata_df)
plsda_vis.interactive()

## 8. Pathway Enrichment Analysis (ORA)

**Over-Representation Analysis (ORA)** translates a list of fluctuating molecules into a biological narrative. In sourdough research, this identifies which microbial "engines" (pathways) are actively driving the fermentation process. Here we gone execute the analysis based on the **sspa** python package [Github Link SSPA](https://github.com/cwieder/py-ssPA)

### **Workflow**

* **Feature Selection:** Significant metabolites are isolated based on **Volcano Plot** thresholds (typically $|log_2FC| > 0.69$ and $-log_{10}(p\text{-value}) > 1.33$), ensuring results are both biologically impactful and statistically robust.
* **Pathway Database:** To determine if a pathway is "over-represented," we compare significant hits against the complete metabolic network of a reference strain. Selecting organism-specific libraries (e.g., ***L. plantarum*** for bacteria, ***S. cerevisiae*** for yeast) ensures the results are taxonomically relevant to the sourdough microbiome.
* **Background Selection:** To avoid analytical bias, the background is defined as the total set of metabolites detected and annotated in the experiment. By comparing significant hits against this measured "universe" (rather than the entire theoretical KEGG database), we ensure that enrichment reflects true biological shifts rather than the detection limits of the LC-MS/MS.
* **Identifier Mapping:** Standardized identifiers (e.g., KEGG IDs) are used to map the measured metabolites to organism-specific libraries (e.g., L. plantarum or S. cerevisiae). This ensures the results are taxonomically relevant to the sourdough microbiome. (This step makes sure the pathway databse and the annotation of our LC-MS/MS data contain the same identifiers). In our case we gone use the InchIkeys for both datasets.
* **Statistical Enrichment:** A hypergeometric test calculates the probability ($p\text{-value}$) that the overlap between your significant hits and a pathway occurred by chance. A lower $p\text{-value}$ suggests the fermentation conditions have specifically "turned up" that metabolic pathway.
* **Visualization:** Results are displayed via dot plots or bar charts, using $-log_{10}(p\text{-value})$ as the primary enrichment score to highlight high-confidence pathways.

### **Notes**

* **Multi-Kingdom Complexity:** Because sourdough is a consortium, an enriched pathway (e.g., *Amino acid metabolism*) may be driven by LAB, yeast, or both. Cross-referencing results against different species libraries is essential.
* **Analytical Bias:** LC-MS/MS is naturally biased toward specific chemical classes. If a pathway appears "silent," it may simply be that those metabolites are not detectable by the current instrument settings.
* **Annotation Bias:** Only annotated features are considered.
* **Biological Meaning:** Enrichment indicates metabolic **potential** and activity shifts; it should be interpreted alongside the fermentation context (e.g., acidification, leavening) to avoid over-interpretation.


### **Step-by-Step**
* **Step 1:** Preparation of our dataframe according to recommendation of sspa package.
* **Step 2:** Preparation of the background metabolite which corresponds to the entire metabolite space measured (including non enriched metabolites).
* **Step 3:** Download the strain specific pathway database (KEGG)
* **Step 4:** Identifier harmonization, for standardization and mapping of annotated features and the metabolite names in the pathway database. Here KEGG IDs are converted to InchIkeys for compatibility.
* **Step 5:** Statistical Evaluation of Pathway Enrichment Pipeline and vizualisation


**8.1: Input Data Preparation**

In [ ]:
class ORADataPreparer:
    def __init__(self, feature_df, metadata_df, inchkeys):
        """
        Prepares metabolomics data for ORA/SSPA analysis.
        feature_df: index=metabolites, columns=samples
        metadata_df: index=samples
        """
        df = feature_df.copy()
        df.index = inchkeys
        df = df[df.index.notnull()]


        self.master_feature = df.copy()
        self.metadata = metadata_df.copy()
        self.processed_df = None

        # 1. Clean data immediately upon initialization
        self._prepare_master_table()

    def _prepare_master_table(self):
        """Internal logic to filter 'mz/' and clean index pipes."""
        # Filter out rows with "mz/"
        df = self.master_feature[~self.master_feature.index.astype(str).str.contains("mz/", na=False)].copy()

        # Clean index: cut away before first "|" and after last "|"
        new_index = []
        for label in df.index.astype(str):
            parts = label.split("|")
            if len(parts) >= 3:
                # Keep everything between the first and last pipe
                new_index.append("|".join(parts[1:-1]))
            else:
                new_index.append(label)

        df.index = new_index
        self.processed_df = df.T

    def show_loading(self):
        """Displays a CSS spinner in the output area."""
        spinner_html = """
        <div style="display: flex; align-items: center; justify-content: center; height: 200px; flex-direction: column;">
            <div class="loader"></div>
            <p style="font-family: sans-serif; margin-top: 15px;">Processing and storing data... Please wait.</p>
            <style>
                .loader {
                  border: 8px solid #f3f3f3;
                  border-top: 8px solid #27ae60;
                  border-radius: 50%;
                  width: 50px;
                  height: 50px;
                  animation: spin 1s linear infinite;
                }
                @keyframes spin {
                  0% { transform: rotate(0deg); }
                  100% { transform: rotate(360deg); }
                }
            </style>
        </div>
        """
        with self.output:
            clear_output(wait=True)
            display(HTML(spinner_html))

    def create_interface(self):
        """Generates the interactive widget UI."""
        attr_options = sorted(self.metadata.columns.tolist())

        # Widgets
        self.w_attr = widgets.Dropdown(options=attr_options, description="Attribute:")
        self.w_g1 = widgets.Dropdown(description="Group 1:")
        self.w_g2 = widgets.Dropdown(description="Group 2:")
        self.btn_store = widgets.Button(
            description="Store Tables to ORA_df and ORA_metadata",
            button_style='success',
            icon='database',
            layout=widgets.Layout(width='650px')
        )
        self.output = widgets.Output()

        # Logic for dropdown dependencies
        def update_groups(*args):
            attr = self.w_attr.value
            groups = sorted([str(x) for x in self.metadata[attr].unique() if pd.notnull(x)])
            self.w_g1.options = groups
            self.w_g2.options = groups
            if len(groups) > 1:
                self.w_g1.value = groups[0]
                self.w_g2.value = groups[1]

        def on_click_store(b):
            # 1. Show Spinner
            self.show_loading()

            # 2. Run computation
            with self.output:
                attr = self.w_attr.value
                g1 = self.w_g1.value
                g2 = self.w_g2.value

                if g1 == g2:
                    clear_output()
                    print("Error: Please select two different groups.")
                    return

                # Filter metadata for selected groups
                selected_meta = self.metadata[self.metadata[attr].astype(str).isin([g1, g2])][[attr]].copy()

                # Align feature table with selected metadata samples
                final_df = self.processed_df.loc[self.processed_df.index.intersection(selected_meta.index)]

                # Filtering, Imputation, and Scaling
                df_filt = final_df.loc[:, final_df.isin([' ', np.nan, 0]).mean() < 0.5]
                imputed_mat = df_filt.fillna(df_filt.median())
                log2_mat = np.log2(imputed_mat.replace(0, 1e-9))
                processed_data = (log2_mat - log2_mat.mean(axis=0)) / log2_mat.std(axis=0)

                # Global assignment
                globals()['ORA_df'] = processed_data
                globals()['ORA_metadata'] = selected_meta

                # 3. Clear spinner and show results
                clear_output(wait=True)
                print(f"✅ Data stored successfully!")
                print(f"Comparison: {g1} vs {g2}")
                print(f"Shape: {final_df.shape[0]} samples, {final_df.shape[1]} metabolites")
                print("\nFEATURE TABLE (Top 5)")
                print("--------------")
                display(processed_data.head(5))
                print("\nMETADATA TABLE (Top 5)")
                print("--------------")
                display(selected_meta.head(5))

        # Observers
        self.w_attr.observe(update_groups, names='value')
        self.btn_store.on_click(on_click_store)

        # Initial trigger and display
        update_groups()
        display(widgets.VBox([
            widgets.HTML("<b>ORA Data Preparation Tool</b>"),
            self.w_attr,
            widgets.HBox([self.w_g1, self.w_g2]),
            self.btn_store,
            self.output
        ]))
preparer = ORADataPreparer(feature_df, metadata_df,inchkeys)
preparer.create_interface()

**8.2 Background List**

In [ ]:
class GenerateBackgroundSet:
    def __init__(self, inchkeys):
        """
        Cleans and stores a background set of InChIKeys.
        Input: list, Series, or Index of InChIKeys.
        """
        self.output = widgets.Output()
        # Ensure we are working with a pandas Series for easy filtering
        s = pd.Series(inchkeys)

        # Filter: Drop NaN, None, empty strings, or strings that are just whitespace
        self.background = s.dropna().loc[lambda x: (x.astype(str).str.strip() != "")].unique().tolist()

    def show_loading(self):
        """Displays the CSS spinner in the output area."""
        spinner_html = """
        <div style="display: flex; align-items: center; justify-content: center; height: 150px; flex-direction: column;">
            <div class="loader"></div>
            <p style="font-family: sans-serif; margin-top: 15px;">Generating Background Set...</p>
            <style>
                .loader {
                  border: 6px solid #f3f3f3;
                  border-top: 6px solid #9b59b6;
                  border-radius: 50%;
                  width: 40px;
                  height: 40px;
                  animation: spin 1s linear infinite;
                }
                @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
            </style>
        </div>
        """
        with self.output:
            clear_output(wait=True)
            display(HTML(spinner_html))

    def get_set(self):
        """Returns the cleaned list and displays a summary."""
        self.show_loading()

        # Simulate processing time for UI consistency
        import time
        time.sleep(0.5)

        with self.output:
            clear_output(wait=True)
            print(f"✅ Background set generated successfully!")
            print(f"Total unique, valid InChIKeys: {len(self.background)}")

        display(self.output)
        return list(set(self.background))
bg_inchikeys = GenerateBackgroundSet( inchkeys )
background_list = bg_inchikeys.get_set()

Output()

**8.3: Download Strain Pathway**





In [ ]:
class BackgroundDatabase:
    def __init__(self):
        # Master dictionary to keep track of all downloaded strains
        self.BackgroundNetwork = {}
        self.output = widgets.Output()

        self.strain_options = {
            'BACTERIA: Fructilactobacillus sanfranciscensis (lsn)': 'lsn',
            'BACTERIA: Levilactobacillus brevis (lbr)': 'lbr',
            'BACTERIA: Lactiplantibacillus plantarum (lpl)': 'lpl',
            'BACTERIA: Companilactobacillus paralimentarius (lali)': 'lali',
            'BACTERIA: Lacticaseibacillus paracasei (lca)': 'lca',
            'BACTERIA: Limosilactobacillus reuteri (lre)': 'lre',
            'BACTERIA: Companilactobacillus crustorum (lct)': 'lct',
            'YEAST: Saccharomyces cerevisiae (sce)': 'sce',
            'YEAST: Pichia kudriavzevii (pkz)': 'pkz',
            'YEAST: Kazachstania servazzi (kaf)': 'kaf',
            'YEAST: Kazachstania humilis (khu)': 'khu'
        }

    def download_KEGG_fixed(self, organism, filepath=None):

        # 1. Get pathway list
        url = f'http://rest.kegg.jp/list/pathway/{organism}'
        data = requests.get(url)
        pathways = [p.split('\t') for p in data.text.split('\n') if p]
        pathway_dict = {p[0]: p[1] for p in pathways}

        pathway_compound_mapping = {}
        base_url = 'http://rest.kegg.jp/get/'

        # 2. Get compounds for each pathway
        for pathid in tqdm(list(pathway_dict.keys()), desc=f"Mapping {organism}"):
            page = requests.get(base_url + pathid)
            lines = page.text.split("\n")

            complist = []
            in_compound_section = False

            for line in lines:
                if not line.strip(): continue

                if line.startswith("COMPOUND"):
                    in_compound_section = True
                    parts = line.split()
                    if len(parts) > 1: complist.append(parts[1])
                    continue

                if in_compound_section:
                    if line[0].isupper() and not line.startswith(" "):
                        break
                    match = re.search(r'(C\d{5})', line)
                    if match: complist.append(match.group(1))

            if complist:
                pathway_compound_mapping[pathid] = list(set(complist))

        # 3. Create DataFrame
        df = pd.DataFrame.from_dict(pathway_compound_mapping, orient='index')
        df.insert(0, 'Pathway_name', df.index.map(pathway_dict.get))

        if filepath:
            release_data = requests.get('http://rest.kegg.jp/info/kegg')
            version_match = re.search(r'Release (\d+\.\d+)', release_data.text)
            version_no = version_match.group(1) if version_match else "unknown"
            fpath = f"{filepath}/KEGG_{organism}_fixed_R{version_no}.gmt"
            df.to_csv(fpath, sep="\t", header=False, index=True)
            print(f"Success! Saved to: {fpath}")

        return df

    def download_networks(self, selected_code):
        """Downloads and stores the selected KEGG network."""
        with self.output:
            clear_output(wait=True)
            print(f"📥 Downloading network for: {selected_code}...")
            try:
                pathway_df = self.download_KEGG_fixed(selected_code, filepath=".")

                # Store in internal dictionary
                self.BackgroundNetwork[selected_code] = pathway_df

                # Assign to global variables
                globals()['BackgroundNetwork'] = self.BackgroundNetwork
                globals()['Kegg_Pathway'] = pathway_df

                print("✅ KEGG Pathway (saved in the variable Kegg_Pathway)")
                metabolite_matrix = pathway_df.iloc[:, 1:]
                flat_list = metabolite_matrix.values.flatten()
                cleaned_list = [m for m in flat_list if m is not None and pd.notna(m)]
                unique_metabolites = list(set(cleaned_list))
                n_metabolites = len(unique_metabolites)
                n_pathways = len(pathway_df.index)



                print(f"Total Number of Pathways: {n_pathways}")
                print(f"Total Number of Metabolites: {n_metabolites}")

                display(pathway_df.head(5))

            except Exception as e:
                print(f"❌ Error downloading {selected_code}: {e}")

    def interactive(self):
        # Dropdown Widget
        self.w_dropdown = widgets.Dropdown(
            options=self.strain_options,
            value='lpl',
            description='Select Strain:',
            layout=widgets.Layout(width='90%')
        )

        # Large Bold Button
        btn_download = widgets.Button(
            description="DOWNLOAD STRAIN NETWORK",
            button_style='success',
            icon='cloud-download',
            layout=widgets.Layout(width='90%', height='55px', margin='10px 0px 10px 0px')
        )
        btn_download.style.font_weight = 'bold'

        def on_click_download(b):
            self.download_networks(self.w_dropdown.value)

        btn_download.on_click(on_click_download)

        # Simplified UI without the green box/border
        ui = widgets.VBox([
            widgets.HTML("<h2>KEGG Pathway Downloader</h2>"),
            widgets.HTML("<i>Select a strain and click download to add it to your background library.</i>"),
            self.w_dropdown,
            btn_download,
            self.output
        ])

        display(ui)


bg_tool = BackgroundDatabase()
bg_tool.interactive()

**Step 8.4: Identifier Harmonization**

In [ ]:
import pandas as pd
from datasets import load_dataset, utils

class KEGGPathwayIdentifierHarmonization:
    def __init__(self, dataset_path="https://raw.githubusercontent.com/LauraElCam/FoodBiotech_Laboratory_Course_Sourdough/main/kegg_inchikey_mapping.csv"):
        self.dataset_path = dataset_path
        self._load_conversion_data()

    def _load_conversion_data(self):
        """Loads the InChIKey mapping from Hugging Face quietly."""
        if 'my_kegg_cache' not in globals():
            # Change df_kegg to df_mapping here
            df_mapping = pd.read_csv(self.dataset_path)
            df_mapping = df_mapping.fillna('None')
            # Now df_mapping is defined and ready to be zipped
            globals()['my_kegg_cache'] = dict(zip(df_mapping['KEGG ID'], df_mapping['InChIKey14']))

        self.kegg_dict = globals()['my_kegg_cache']

    @staticmethod
    def _push_none_to_end(row):
        """Helper to shift all valid data to the left and None/empty to the right."""
        row_list = row.tolist()
        # Filter out anything that looks like None or is empty
        valid_data = [x for x in row_list if x is not None and str(x).strip() != "" and str(x).lower() != "none"]
        # Pad with None to maintain original row length
        return pd.Series(valid_data + [None] * (len(row_list) - len(valid_data)))

    def run_harmonization(self, df_input, version="117.0"):
        """Performs the full transformation on the Kegg_Pathway table."""

        # 1. Replace KEGG IDs with InChIKeys using the dictionary
        # This scans the entire table and swaps IDs where found
        df_transformed = df_input.replace(self.kegg_dict)

        # 2. Shift all None values to the end of each row
        # This ensures the matrix is left-justified
        df_clean = df_transformed.apply(self._push_none_to_end, axis=1)
        df_clean.columns = df_input.columns
        df_clean = df_clean.fillna("None")

        # 3. Calculate Statistics
        n_pathways = len(df_clean)
        # Flatten and count unique values excluding "None"
        flat_data = df_clean.values.flatten()
        unique_metabolites = len(set([m for m in flat_data if str(m).lower() != "none"]))

        # 4. Save to file (GMT style)
        # Assuming the organism code 'lpl' is inferred from your context
        filename = f"./KEGG_lpl_fixed_R{version}.gmt"
        df_clean.to_csv(filename, sep="\t", header=False, index=True)

        # 5. Global Output and Display
        print(f"Success! Saved to: {filename}")
        print(f"KEGG ID converted to InchiKeys")
        print(f"✅ KEGG Pathway (saved in the variable Kegg_Pathway)")
        print(f"Total Number of Pathways: {n_pathways}")
        print(f"Total Number of Metabolites: {unique_metabolites}")

        # Overwrite the global Kegg_Pathway variable with the harmonized version
        globals()['Kegg_Pathway'] = df_clean

        return df_clean

# --- Usage ---
# Initialize the harmonizer
harmonizer = KEGGPathwayIdentifierHarmonization()

# Run the harmonization on your existing Kegg_Pathway dataframe
Kegg_Pathway = harmonizer.run_harmonization(Kegg_Pathway)

# Display the final harmonized table
Kegg_Pathway.head()

Success! Saved to: ./KEGG_lpl_fixed_R117.0.gmt
KEGG ID converted to InchiKeys
✅ KEGG Pathway (saved in the variable Kegg_Pathway)
Total Number of Pathways: 88
Total Number of Metabolites: 2571


,Pathway_name,0,1,2,3,4,5,6,7,8,...,138,139,140,141,142,143,144,145,146,147
lpl00010,Glycolysis / Gluconeogenesis - Lactiplantibaci...,NBSCHQHZLSJFNQ,KHPXUQMNIQBQEV,GNGACRATGGDKBX,ZSLZBFCDCINBPY,IKHGUXGNUITLKF,NGFMICBWJRZIBI,FBHYCDOVYMVLEN,RNBGYGVWRKECFJ,JVTAAEKCZFNVCJ,...,None,None,None,None,None,None,None,None,None,None
lpl00020,Citrate cycle (TCA cycle) - Lactiplantibacillu...,KHPXUQMNIQBQEV,ZSLZBFCDCINBPY,KDYFGRWQOYBRFD,ZWUKRGPVMMTMAF,DTBNBXWJWCWCIK,VNOYUJKHFWYWIR,BJEPYKJPYRNKOW,LCTONWCANYUPML,ODBLHEXUDAPZAU,...,None,None,None,None,None,None,None,None,None,None
lpl00030,Pentose phosphate pathway - Lactiplantibacillu...,NBSCHQHZLSJFNQ,BIRSGZKFKXLSJQ,UZYFNQCWJLIAKE,RGHNJXZEOKUKBD,MNQZXJOMYWMBOU,FNZLKVNUWIIPSJ,YAHZABJORDUQGO,OVPRPPOVAXRCED,RNBGYGVWRKECFJ,...,None,None,None,None,None,None,None,None,None,None
lpl00040,Pentose and glucuronate interconversions - Lac...,HEBKCHPVOIAQTA,HEBKCHPVOIAQTA,GNGACRATGGDKBX,RGHNJXZEOKUKBD,WGCNASOHLSPBMP,HEBKCHPVOIAQTA,FNZLKVNUWIIPSJ,C20680,UQIGQRSJIKIPKZ,...,None,None,None,None,None,None,None,None,None,None
lpl00051,Fructose and mannose metabolism - Lactiplantib...,VFRROHXSMXFLSN,GNGACRATGGDKBX,SHZGCJCMOBCMKK,GACTWZZMVMUKNG,KNYGWWDTPGSEPD,SHZGCJCMOBCMKK,BSABBBMNWQWLLU,MNQZXJOMYWMBOU,QZNPNKJXABGCRC,...,None,None,None,None,None,None,None,None,None,None


**8.5: ORA (Overrepresentation Analysis**

In [ ]:
class ORAVisualizer:
    def __init__(self, ora_df, metadata, kegg_pathway, background):
        self.ora_df = ora_df
        self.metadata = metadata
        self.kegg_pathway = kegg_pathway
        self.background = background
        self.ora_res = None
        self.output = widgets.Output()

        # UI Elements
        self.slider = widgets.IntSlider(
            value=20, min=5, max=200, step=1,
            description='Top N:',
            continuous_update=False,
            layout=widgets.Layout(width='50%')
        )
        self.slider.observe(self._on_value_change, names='value')

    def show_loading(self):
        """Displays a CSS spinner during heavy calculations."""
        spinner_html = """
        <div style="display: flex; align-items: center; justify-content: center; height: 200px; flex-direction: column;">
            <div class="loader"></div>
            <p style="font-family: sans-serif; margin-top: 15px;">Running Analysis... Please wait.</p>
            <style>
                .loader {
                  border: 8px solid #f3f3f3;
                  border-top: 8px solid #3498db;
                  border-radius: 50%;
                  width: 50px;
                  height: 50px;
                  animation: spin 1s linear infinite;
                }
                @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
            </style>
        </div>
        """
        with self.output:
            clear_output(wait=True)
            display(HTML(spinner_html))

    def _run_analysis(self):
        """Executes the sspa calculation with a loading state."""
        self.show_loading()

        ora_obj = sspa.sspa_ora(
            mat=self.ora_df,
            metadata=self.metadata,
            pathways=self.kegg_pathway,
            DA_testtype='ttest',
            DA_cutoff=0.05,
            custom_background=self.background
        )

        ora_res = ora_obj.over_representation_analysis()
        ora_res = ora_res.sort_values(by="P-value",ascending=True)

        # Ensure 'Pathway_name' exists for plotting; use index if not present
        if 'Pathway_name' not in ora_res.columns:
            ora_res['Pathway_name'] = ora_res.index
        self.ora_res = ora_res
        self.display_results()

    def _on_value_change(self, change):
        self.display_results()

    def display_results(self):
        """Creates the horizontal bar plot and table layout."""
        if self.ora_res is None:
            return

        top_n = self.slider.value
        top_df = self.ora_res.copy()
        top_df = top_df.drop(columns=['P-adjust'])
        top_df = top_df.sort_values(by="P-value",ascending=True).head(top_n).copy()
        top_dfp = top_df.copy()
        top_dfp['P-value'] = -np.log10(top_dfp['P-value'])

        with self.output:
            clear_output(wait=True)

            # Updated Plotting Logic
            plt.figure(figsize=(8, top_n * 0.3 + 2))
            sns.barplot(
                data=top_dfp,
                y="Pathway_name",
                x="P-value",
                orient="h",
                hue='Pathway_name',
                palette="viridis"
            )
            plt.title(f"Top {top_n} Pathways by P-value", fontweight='bold')
            plt.xlabel("-log10(P-value)")
            plt.ylabel("Pathway")
            plt.grid(axis='x', linestyle=':', alpha=0.6)

            plt.show()
            display(top_df)

    def show(self):
        """Initial display and trigger analysis."""
        display(widgets.VBox([
            widgets.HTML("<h3>Reactive ORA Results</h3>"),
            self.slider,
            self.output
        ]))
        # Run analysis after the output widget is displayed so the spinner is visible
        self._run_analysis()

# --- Execution ---
# Note: Ensure ORA_df, ORA_metadata, Kegg_Pathway, and background_list are defined
visualizer = ORAVisualizer(
    ora_df=ORA_df,
    metadata=ORA_metadata.iloc[:, 0],
    kegg_pathway=Kegg_Pathway,
    background=background_list
)
visualizer.show()

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:611: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)
